In [ ]:
import time
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
import requests
import pandas as pd

def fetch_wind(name, longitude, latitude, start_date, end_date):
    url = "https://archive-api.open-meteo.com/v1/archive"

    session = requests.Session()
    retries = Retry(total=5, backoff_factor=10, status_forcelist=[500, 502, 503, 504])
    session.mount("https://", HTTPAdapter(max_retries=retries))

    params = {
        "latitude": latitude,
        "longitude": longitude,
        "start_date": start_date,
        "end_date": end_date,
        "hourly": "wind_speed_100m,wind_direction_100m",
        "timezone": "Pacific/Auckland"
    }

    response = session.get(url, params=params, timeout=60)
    response.raise_for_status()
    data = response.json()

    df = pd.DataFrame({
        "time":               data["hourly"]["time"],
        "wind_speed_kmh":     data["hourly"]["wind_speed_100m"],
        "wind_direction_deg": data["hourly"]["wind_direction_100m"]
    })

    df["time"] = pd.to_datetime(df["time"])
    print(df.head(10))
    print(f"\nShape: {df.shape}")
    print(f"Missing values: {df.isnull().sum().sum()}")

    df = df.rename(columns={
        "wind_speed_kmh":     f"{name}_wind_kmh",
        "wind_direction_deg": f"{name}_wind_dir_deg"
    })
    return df

In [18]:
# Locations chosen to match NZ's major wind generation clusters.
# Coordinates verified against Wikipedia / official sources.
#
#              name,                  longitude,   latitude,  start_date,   end_date
request =  [['Palmerston_North'     , 175.61,  -40.36, "2014-01-01", "2026-05-25"],  # Hub of NZ wind: Tararua, Te Apiti, Turitea, Waipipi
            ['Wellington'           , 174.78,  -41.29, "2014-01-01", "2026-05-25"],  # West Wind 143 MW
            ['Harapaki_HawkesBay'   , 176.69,  -39.18, "2014-01-01", "2026-05-25"],  # Harapaki 176 MW — NZ 2nd largest
            ['Te_Uku_Waikato'       , 174.96,  -37.88, "2014-01-01", "2026-05-25"],  # Te Uku 64 MW near Raglan
            ['Kaiwera_Downs_Southland', 169.06, -46.24, "2014-01-01", "2026-05-25"]] # Kaiwera Downs 198 MW (growing)

wind_data = pd.DataFrame()
for name, lon, lat, start, end in request:
    df = fetch_wind(name, lon, lat, start, end)
    time.sleep(5)

    if wind_data.empty:
        wind_data = df
    else:
        wind_data = wind_data.merge(df, on="time", how="outer")

print(wind_data.head(5))

KeyboardInterrupt: 

In [15]:
wind_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 108672 entries, 0 to 108671
Data columns (total 11 columns):
 #   Column                                Non-Null Count   Dtype         
---  ------                                --------------   -----         
 0   time                                  108672 non-null  datetime64[ns]
 1   Palmerston_North_wind_kmh             108672 non-null  float64       
 2   Palmerston_North_wind_dir_deg         108672 non-null  int64         
 3   Wellington_wind_kmh                   108672 non-null  float64       
 4   Wellington_wind_dir_deg               108672 non-null  int64         
 5   Harapaki_HawkesBay_wind_kmh           108672 non-null  float64       
 6   Harapaki_HawkesBay_wind_dir_deg       108672 non-null  int64         
 7   Te_Uku_Waikato_wind_kmh               108672 non-null  float64       
 8   Te_Uku_Waikato_wind_dir_deg           108672 non-null  int64         
 9   Kaiwera_Downs_Southland_wind_kmh      108672 non-null  floa

In [ ]:
wind_data.to_csv("Wind_data_100m.csv", index=False)